### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(
    model="qwen/qwen3.8-27b"
)

In [8]:

from langchain.agents import create_agent
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    ModelRetryMiddleware,
)
from langchain.tools import tool


@tool
def get_weather(location: str) -> str:
    """Return mock weather details for the given location."""
    return f"The weather in {location} is sunny."


agents = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=(
        "You are a helpful travel assistant. "
        "Always use the weather tool when the user asks about the weather."
    ),
    middleware=[
        ModelRetryMiddleware(
            max_retries=3,
            initial_delay=1.0,
            backoff_factor=2.0,
        ),
        ModelCallLimitMiddleware(run_limit=3),
        # ModelCallLimitMiddleware(run_limit=1),
    ],
)

response = agents.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the weather in Akola?",
            }
        ]
    }
)

print(response["messages"][-1].content)


The weather in Akola is currently sunny. Let me know if you’d like more details or forecasts for other locations!


In [9]:

import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import (
    ModelRetryMiddleware,
    ModelCallLimitMiddleware,
    wrap_model_call,
)

load_dotenv()


# 1. Configure the Groq model
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"],
)


# 2. Create a mock weather tool
@tool
def get_weather(location: str) -> str:
    """Return mock weather information for a location."""
    print(f"[TOOL] get_weather called for: {location}")
    return f"The weather in {location} is sunny."


# 3. Create logging middleware
@wrap_model_call
def log_model_call(request, handler):
    """Log model-call activity and any errors."""
    print("[LOG] Calling model...")

    try:
        response = handler(request)
        print("[LOG] Model call succeeded.")
        return response
    except Exception as exc:
        print(f"[LOG] Model call failed: {type(exc).__name__}: {exc}")
        raise


# 4. Configure built-in middleware
middleware = [
    ModelRetryMiddleware(
        max_retries=3,
        initial_delay=1.0,
        backoff_factor=2.0,
    ),
    ModelCallLimitMiddleware(run_limit=5),
    log_model_call,
]


# 5. Create the agent
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=(
        "You are a helpful travel assistant. "
        "Always use the get_weather tool for weather questions. "
        "After receiving the tool result, answer the user."
    ),
    middleware=middleware,
)


# 6. Run the agent
if __name__ == "__main__":
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "What is the weather in Akola?",
                }
            ]
        }
    )

    print("\n--- Final Answer ---")
    print(result["messages"][-1].content)


[LOG] Calling model...
[LOG] Model call failed: NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
[LOG] Calling model...
[LOG] Model call failed: NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
[LOG] Calling model...
[LOG] Model call failed: NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
[LOG] Calling model...
[LOG] Model call failed: NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [16]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
)

checkpointer = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=checkpointer,
    system_prompt="You are a reliable AI assistant.",
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 200),
            keep=("messages", 10),
        )
    ],
)

config = {"configurable": {"thread_id": "summary-test"}}

# Send multiple messages to build conversation history
for i in range(8):
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        f"Message {i + 1}: My project is Project Alpha. "
                        f"Here are some details about the project: "
                        f"We are building a customer management application. "
                        f"It includes reporting, user management, testing, "
                        f"automation, API integration, and data validation. "
                        * 3
                    ),
                }
            ]
        },
        config=config,
    )

    print(f"Completed message {i + 1}")

# Inspect the final checkpointed state
state = agent.get_state(config)

print("\n--- Current message history ---")
for message in state.values.get("messages", []):
    print(
        f"{message.type}: "
        f"{str(message.content)[:150]}"
    )

Completed message 1
Completed message 2
Completed message 3
Completed message 4
Completed message 5
Completed message 6
Completed message 7
Completed message 8

--- Current message history ---
human: Here is a summary of the conversation to date:

## SESSION INTENT

The user is initiating a session for "Project Alpha," a customer management applica
ai: I have noted the details for **Project Alpha** once more. The context remains unchanged:

*   **Project:** Project Alpha
*   **Type:** Customer Manage
human: Message 4: My project is Project Alpha. Here are some details about the project: We are building a customer management application. It includes report
ai: I have received the details for **Project Alpha** again. The information is consistent with previous messages:

*   **Project:** Project Alpha
*   **T
human: Message 5: My project is Project Alpha. Here are some details about the project: We are building a customer management application. It includes report
ai: I have received the

## Fracdtion 

In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
)

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~244 tokens (0.1906%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='57e3bc33-1b4a-415f-901a-595ac766fbf3'), AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants hotels in Paris. Use function search_hotels with city "Paris".', 'tool_calls': [{'id': 'fc_e50dcd4b-1f1c-4fa9-86cb-5afbebf4db87', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 120, 'total_tokens': 165, 'completion_time': 0.095716117, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.00451199, 'prompt_tokens_details': None, 'queue_time': 0.335113158, 'total_time': 0.100228107}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_1d982b31b2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12496-8dc6-7360-ae27-43b1cd7883b4-0'

### Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [37]:

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv()

llm = init_chat_model("groq:qwen/qwen3.8-27b")


@tool("read_balance", description="Check account balance")
def read_balance(account_id: str) -> str:
    """Read user balance."""
    return f"Account {account_id} balance: $5,000"


@tool("transfer_funds", description="Transfer money between accounts")
def transfer_funds(
    account_id: str,
    recipient_account_id: str,
    amount: float,
) -> str:
    """Transfer funds between accounts."""
    return (
        f"Successfully transferred ${amount} "
        f"from {account_id} to {recipient_account_id}."
    )


checkpointer = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[read_balance, transfer_funds],
    checkpointer=checkpointer,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"transfer_funds": True}
        )
    ],
)

config = {"configurable": {"thread_id": "tx-456"}}

# Step 1: Request a transfer
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Transfer $500 from ACC123 to ACC456"
                ),
            }
        ]
    },
    config,
)
print(result["messages"][-1])


content='' additional_kwargs={'tool_calls': [{'id': 'qtqsym30j', 'function': {'arguments': '{"account_id":"ACC123","amount":500,"recipient_account_id":"ACC456"}', 'name': 'transfer_funds'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 370, 'total_tokens': 433, 'completion_time': 0.172069281, 'completion_tokens_details': None, 'prompt_time': 0.026003701, 'prompt_tokens_details': None, 'queue_time': 0.046240867, 'total_time': 0.198072982}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a124b5-2ba5-7ec0-ae8e-9087bb546450-0' tool_calls=[{'name': 'transfer_funds', 'args': {'account_id': 'ACC123', 'amount': 500, 'recipient_account_id': 'ACC456'}, 'id': 'qtqsym30j', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 370, 'output_tokens': 63, 'total_tokens': 433}


In [38]:


resumed = agent.invoke(
        Command(
            resume={
                "decisions": [{"type": "approve"}]
            }
        ),
        config,
    )
print(resumed["messages"][-1].content)


The transfer was successful: **$500.00** has been moved from **ACC123** to **ACC456**.


In [49]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# Mock tools
@tool
def read_email_tool(email_id: str) -> str:
    """Read an email by its ID."""
    return f"Email content for ID: {email_id}"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {recipient} with subject '{subject}'"


# Create agent
agent = create_agent(
    model=llm,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ],
)

config = {"configurable": {"thread_id": "tx-456"}}

# Step 1: Request an email
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Send an email to test@example.com "
                    "with subject 'Meeting' and body 'See you tomorrow.'"
                ),
            }
        ]
    },
    config=config,
)

print("Initial result:", result)




Initial result: {'messages': [HumanMessage(content="Send an email to test@example.com with subject 'Meeting' and body 'See you tomorrow.'", additional_kwargs={}, response_metadata={}, id='b69e8364-2062-41a1-b134-fa24e5e54fc8'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '7vc2e8376', 'function': {'arguments': '{"body":"See you tomorrow.","recipient":"test@example.com","subject":"Meeting"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 374, 'total_tokens': 431, 'completion_time': 0.170056309, 'completion_tokens_details': None, 'prompt_time': 0.026376816, 'prompt_tokens_details': None, 'queue_time': 0.045756684, 'total_time': 0.196433125}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a124ba-b9ca-72b3-89a3-e30fdcf0b2ac-0', tool_calls=[{'name': 

In [50]:
result = agent.invoke(
        Command(
            resume={
                "decisions": [{"type": "approve"}]
            }
        ),
        config=config,
    )
print("Final result:", result["messages"][-1].content)

Final result: Email sent successfully to test@example.com with the subject "Meeting" and body "See you tomorrow."
